In [ ]:
import os
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reproduce.py").is_file())
os.chdir(ROOT)
OUTPUT_DIR = ROOT / "working/analysis/stats-rework/out/trajectories"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
import pandas as pd
import numpy as np
from matplotlib.ticker import LogFormatter
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import json

In [ ]:
# Assign Data to path

PROJECT_PATH = ROOT / 'working/figures/Figure 3/A - survival'
file= 'SurvivalData.xlsx'
PROJECT_PATH


In [ ]:
# Read the Excel file into a DataFrame
all_sheets_data = pd.read_excel(PROJECT_PATH / file, sheet_name=None, engine='openpyxl')
print("Sheet names:", all_sheets_data.keys())


In [ ]:
# Specify the selected sheet name
selected_evolution = 'PLAC'  # Replace 'PC' with your desired sheet name

# Extract the data for the selected sheet
if selected_evolution in all_sheets_data:
    select_survival_data = all_sheets_data[selected_evolution]
    print("Data from the selected sheet:")
    print(select_survival_data.head())  # Display the first few rows
else:
    print(f"Sheet '{selected_evolution}' does not exist in the file.")


In [ ]:
filtered_data = select_survival_data[~select_survival_data['Culture'].isin(['A', 'B', 'C', 'D'])]


In [ ]:
# Filter the DataFrame to include only rows where Strain is 'MG1655'
data = filtered_data[filtered_data['Strain'] == 'MG1655']




In [ ]:
# Add a new column 'treated' to indicate 'before' or 'after'
data['treated'] = data['Day'].apply(
    lambda x: 'before' if x.is_integer() else 'after'
)
data.loc[data['treated'] == 'after', 'Day'] -= 0.5

# Display the updated DataFrame
print(data.tail())


In [ ]:
# Group by the specified columns
grouped = data.groupby(['Drug', 'Day', 'Strain', 'Culture'])

# Calculate percent survival
def calculate_percent_survival(group):
    # Separate 'before' and 'after' data
    before = group[group['treated'] == 'before']
    after = group[group['treated'] == 'after']
    
    if not before.empty and not after.empty:
        # Take the first value of CFU from 'before' for calculation
        before_cfu = before['CFU'].iloc[0]
        # Add the percent survival column to the 'after' group
        after['PercentSurvival'] = (after['CFU'] / before_cfu) * 100
    
    return pd.concat([before, after])

# Apply the function to each group
result = grouped.apply(calculate_percent_survival).reset_index(drop=True)

# Display the updated DataFrame
print(result.head())


In [ ]:
# Filter the result to include only rows where 'treated' is 'after'
result = result[result['treated'] == 'after']
result = result.sort_values(by=["Culture", "Day"])

# Display the filtered DataFrame
print(result.head())
result

In [ ]:
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Nimbus Roman"
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Nimbus Roman"



# Define a color mapping for the drugs
drug_colors = {
    "Levofloxacin": "red",
    "Amikacin": "blue",
    "Cefepime": "forestgreen"
}

# Calculate the mean PercentSurvival for each Day and Drug
mean_data = result.groupby( 'Day')['PercentSurvival'].mean().reset_index()

# Initialize the plot
plt.figure(figsize=(20, 8))

# Plot individual PercentSurvival data
for culture in result['Culture'].unique():
    culture_data = result[result['Culture'] == culture]
    plt.plot(
        culture_data['Day'],
        culture_data['PercentSurvival'],
        color="gray",
        linewidth=5,
        alpha=0.4,
        label=None  # Avoid duplicate legend entries for cultures
    )

# 1. Plot all mean data in purple
plt.plot(
    mean_data['Day'],
    mean_data['PercentSurvival'],
    color="forestgreen",
    linewidth=10,
    label="Mean - All Days (Purple)"
)

# 2. Plot mean data for days < 32 in blue
plt.plot(
    mean_data.loc[mean_data['Day'] < 32, 'Day'],
    mean_data.loc[mean_data['Day'] < 32, 'PercentSurvival'],
    color="blue",
    linewidth=10,
    label="Mean - Days < 32 (Blue)"
)

# 3. Plot mean data for days < 17 in red
plt.plot(
    mean_data.loc[mean_data['Day'] < 17, 'Day'],
    mean_data.loc[mean_data['Day'] < 17, 'PercentSurvival'],
    color="red",
    linewidth=10,
    label="Mean - Days < 17 (Red)"
)



# Set the y-axis to a log10 scale
plt.yscale('log')
plt.ylim(0.0005, 3000)  # Set y-axis limits
plt.yticks([0.001,0.01, 0.1, 1, 10, 100, 1000], ["0.001","0.01", "0.1", "1", "10", "100", "1000"], fontsize=35)
plt.ylabel("Survival (%)", fontsize=40)





# Set x-axis ticks to show every 4 days
every_four_days = result['Day'].unique()[result['Day'].unique() % 15 == 0]  # Filter multiples of 4
plt.xticks(every_four_days, every_four_days.astype(int), fontsize=35)
plt.xlabel("Day", fontsize=40)

plt.xlim(mean_data['Day'].min(), mean_data['Day'].max())
# Adjust the thickness of major and minor tick marks
plt.tick_params(axis='both', which='major', length=12, width=3)  # Major ticks thicker
plt.tick_params(axis='both', which='minor', length=8, width=2)  # Minor ticks thicker

# Increase spine widths
ax = plt.gca()  # Get the current axes
for spine_name, spine in ax.spines.items():
    if spine_name in ["top", "right"]:
        spine.set_visible(False)  # Hide the top and right spines
    else:
        spine.set_linewidth(3)  # Set spine width to 3 for bottom and left spines

# Add legend
#plt.legend(fontsize=15)

# Add gridlines for better readability


# Set the overall plot title
#plt.title("Percent Survival Over Time by Drug", fontsize=25)

# Save the figure
plt.savefig(OUTPUT_DIR / f'PLAC_ps_comb.png', dpi=600, bbox_inches='tight')

# Show the plot
plt.show()
